# m21 lab · Image integration

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Image integration brings a CT or MRI anatomy into the mapping system's coordinates. You segment a synthetic contrast-filled chamber by thresholding, score segmentations by overlap, register landmarks by least squares, register surfaces by iterative closest point, and see why a small surface distance does not guarantee a small error at a target.

| Part | What you do | Equation |
| --- | --- | --- |
| A | Threshold segmentation of a synthetic CT | eq-m21-threshold |
| B | Overlap: Dice and Jaccard | eq-m21-dice |
| C | Landmark registration | eq-m21-procrustes |
| D | Surface registration by ICP | eq-m21-icp |
| E | What a surface cannot pin down | eq-m21-sliding |

Lengths in mm. The synthetic image (intensities, noise, shapes), landmark errors and rotations are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import registration as RG

## Part A · Threshold segmentation of a synthetic CT

Build a small volume: tissue at intensity 100, a contrast-filled ellipsoidal chamber with a vein at 350, and a separate
bright vessel, all with Gaussian noise. Otsu's threshold maximizes the between-class variance
$\omega_0\omega_1(\mu_0 - \mu_1)^2$ (eq-m21-threshold); then keep the largest connected component.

In [ ]:
rng = np.random.default_rng(21)
h = 1.0
ax = np.arange(-40, 40 + h, h)
X, Y, Z = np.meshgrid(ax, ax, ax, indexing="ij")
chamber = (X / 30) ** 2 + (Y / 22) ** 2 + (Z / 18) ** 2 <= 1
vein = (X > 20) & (X < 38) & (np.hypot(Y - 6, Z) < 5)
vessel = np.hypot(X + 5, Z - 30) < 6                         # a separate bright structure above the chamber
truth = chamber | vein
image = np.where(truth | vessel, 350.0, 100.0) + rng.normal(0, 60, X.shape)
t = RG.otsu_threshold(image.ravel())
seg = image > t
seg_lcc = RG.largest_component(seg)
print(f"Otsu threshold {t:.0f}; Dice with the truth: raw {RG.dice(seg, truth):.3f}, largest component {RG.dice(seg_lcc, truth):.3f}")
from scipy import ndimage
smooth = ndimage.gaussian_filter(image, 1.0)                   # denoise first (1 voxel)
ts = RG.otsu_threshold(smooth.ravel())
seg_s = RG.largest_component(smooth > ts)
print(f"after smoothing: threshold {ts:.0f}, Dice {RG.dice(seg_s, truth):.3f}")

In [ ]:
# Checks
assert 180 < t < 270 and RG.dice(seg_lcc, truth) > RG.dice(seg, truth) and RG.dice(seg_s, truth) > RG.dice(seg_lcc, truth)

## Part B · Overlap: Dice and Jaccard

Dice is $2\lvert A\cap B\rvert/(\lvert A\rvert + \lvert B\rvert)$ and Jaccard $\lvert A\cap B\rvert/\lvert A\cup B\rvert$, related by
$D = 2J/(1 + J)$ (eq-m21-dice). For two Gaussian classes, the expected Dice of any threshold has a closed form; compare
Otsu's choice with the best possible one. When the blood pool fills only a small part of the image, Otsu's threshold sits
closer to the minority class than the Dice-optimal one.

In [ ]:
print(f"Jaccard {RG.jaccard(seg_lcc, truth):.3f}; 2J/(1+J) = {2 * RG.jaccard(seg_lcc, truth) / (1 + RG.jaccard(seg_lcc, truth)):.3f}")
p = truth.mean()
ts = np.linspace(100, 350, 251)
d = RG.gaussian_dice(ts, 100, 350, 60, p)
plt.plot(ts, d); plt.axvline(t, ls="--", c="k"); plt.xlabel("threshold"); plt.ylabel("expected Dice"); plt.show()
print(f"blood fraction {p:.3f}; expected Dice at Otsu {RG.gaussian_dice(t, 100, 350, 60, p):.3f}, best {d.max():.3f} at {ts[np.argmax(d)]:.0f}")

In [ ]:
# Checks
assert d.max() > RG.gaussian_dice(t, 100, 350, 60, p) and ts[np.argmax(d)] > t

## Part C · Landmark registration

With paired landmarks, the least-squares rotation comes from the SVD of their cross-covariance, with a sign fix that
rules out reflections, and the translation from the centroids (eq-m21-procrustes); a scale can be added. Locate four
illustrative landmarks with 3 mm error and register. m55 gives the expected target error in closed form.

In [ ]:
L = np.array([[32, 6, 0], [-25, 10, 8], [-20, -12, -6], [5, -20, 12]], float)       # illustrative landmarks
target = np.array([0.0, 0.0, -18.0])                                                     # a point to reach
R0, t0 = RG.rotation(20, 2) @ RG.rotation(-8, 0), np.array([4.0, -6.0, 3.0])
fre, tre = [], []
for _ in range(2000):
    mapped = L @ R0.T + t0 + rng.normal(0, 3 / np.sqrt(3), L.shape)                       # 3 mm RMS localization error
    R, tt, _ = RG.procrustes(L, mapped)
    fre.append(RG.rms(L @ R.T + tt - mapped))
    tre.append(np.linalg.norm(target @ R.T + tt - (target @ R0.T + t0)))
print(f"mean FRE {np.mean(fre):.2f} mm (what the system reports); RMS TRE at the target {np.sqrt(np.mean(np.square(tre))):.2f} mm")
scaled = 1.1 * L @ R0.T + t0                                                             # a 10% scale error (m17)
for s in (False, True):
    R, tt, ss = RG.procrustes(L, scaled, scale=s)
    print(f"{'similarity' if s else 'rigid':10s}: residual {RG.rms(ss * L @ R.T + tt - scaled):.2f} mm, scale {ss:.3f}")

In [ ]:
# Checks
R, tt, ss = RG.procrustes(L, scaled, scale=True)
assert abs(ss - 1.1) < 1e-9 and RG.rms(ss * L @ R.T + tt - scaled) < 1e-9

## Part D · Surface registration by ICP

Iterative closest point matches each mapped point to the nearest image point, solves the landmark problem for those
pairs, and repeats (eq-m21-icp). The mean squared distance never increases, but ICP converges to the nearest local
minimum, so the starting pose matters; trying several starts is the classic remedy (Besl and McKay).

In [ ]:
surf = RG.ellipsoid_points((30, 22, 18), 3000, rng)
src = surf[rng.choice(3000, 400, replace=False)]
print("start   final RMS distance   RMS error to truth")
for ang in (10, 30, 60, 90, 120, 170):
    moved = src @ RG.rotation(ang, 2).T + np.array([3.0, -2.0, 1.0])
    R, tt, hist = RG.icp(moved, surf, iters=150)
    print(f"{ang:4d}°   {np.sqrt(hist[-1]):8.3f} mm        {RG.rms(moved @ R.T + tt - src):7.2f} mm")
moved = src @ RG.rotation(120, 2).T
best = min((RG.icp(moved, surf, R0=RG.rotation(a, 2), iters=150) for a in range(0, 360, 45)), key=lambda r: r[2][-1])
print("best of 8 starting rotations: RMS error", round(RG.rms(moved @ best[0].T + best[1] - src), 3), "mm")

In [ ]:
# Checks
R, tt, hist = RG.icp(src @ RG.rotation(10, 2).T, surf, iters=150)
assert np.all(np.diff(hist) <= 1e-9) and RG.rms(src @ RG.rotation(10, 2).T @ R.T + tt - src) < 1e-6

## Part E · What a surface cannot pin down

For small motions, the point-to-plane residuals are linear in rotation and translation, and $J^\top J$ shows which motions
the surface constrains (eq-m21-sliding). A sphere can rotate freely about its center: three zero eigenvalues. A target
inside or on the sphere still moves. That is how a small surface distance can hide a large target error.

In [ ]:
for axes in ((25, 25, 25), (30, 22, 18)):
    pnts = RG.ellipsoid_points(axes, 3000, rng)
    ev = np.linalg.eigvalsh(RG.point_to_plane_hessian(pnts, RG.ellipsoid_normals(pnts, axes)))
    print(f"axes {axes}: smallest eigenvalue {ev.min():.2e}, largest {ev.max():.2e}")
sph = RG.ellipsoid_points((25, 25, 25), 500, rng)
rot = sph @ RG.rotation(10, 2).T
print("sphere turned 10 degrees: surface distance", round(float(np.max(np.abs(np.linalg.norm(rot, axis=1) - 25))), 6),
      "mm; point displacement", round(RG.rms(rot - sph), 2), "mm")

In [ ]:
# Checks
pnts = RG.ellipsoid_points((25, 25, 25), 3000, rng)
ev = np.linalg.eigvalsh(RG.point_to_plane_hessian(pnts, RG.ellipsoid_normals(pnts, (25, 25, 25))))
assert np.sum(ev < 1e-8 * ev.max()) == 3

## Where to go next

- Combine landmarks and surface in one cost and see the sliding modes disappear.
- Add the m17 field distortion to the mapped points and register with a similarity transform.
- m55 puts segmentation, registration and error budgets together in a capstone.